# TD3C physical tape (Kpler) vs ICE paper

Running case: **MEG → China VLCC**. Paper is ICE **TDL** / **DFN** in $/t (`notebooks/01_td3c_ffa_positioning.ipynb`). This notebook is the *why*: reachable ships, ballast, fixtures, China waiting, ton-miles.

Read `docs/kpler_signals.html` first. Kpler fixture `$` ≠ Baltic WS ≠ ICE $/t.

Kpler zone for the Gulf is **`Mideast Gulf`** (not “Persian Gulf”). Tonnage matcher: destination = where the ship can *arrive* — here the **load** region. Defaults: `eta=7`, `speed=11` kn (shock speed as a scenario, not a daily default).

Caches parquet under `data/cache/` (gitignored). Set `REFRESH=True` after weekend. Fixture cell is **counts + median $** only — no vessel names.

In [1]:
from datetime import date, timedelta
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT.parent / "desk_core"))

from freight.ice import asof, front_n, load_daily_prompt, load_ffa_monthly
from freight import kpler as kp

pd.options.display.float_format = lambda x: f"{x:,.2f}"
CACHE = ROOT / "data" / "cache"
REFRESH = False
START = date.today() - timedelta(days=180)
ICE_START = date(2024, 1, 1)
print("ROOT", ROOT)

ROOT c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\freight_dashboard


In [2]:
cfg = kp.liquids_config(ROOT)
print("kpler_v1_auth", getattr(cfg, "auth_validated", None))

tdl = load_ffa_monthly("TDL", start=ICE_START, cache_dir=CACHE, refresh=REFRESH)
dfn = load_daily_prompt("DFN", start=ICE_START, cache_dir=CACHE, refresh=REFRESH)
tdl_day, tdl_asof = asof(tdl)
front = front_n(tdl_asof, n=1)
last_dfn = dfn["TRADE_DATE"].max()
prompt = dfn.loc[dfn["TRADE_DATE"] == last_dfn]
tdl_px = float(front.iloc[0]["SETTLE_USD_T"]) if len(front) else float("nan")
dfn_px = float(prompt.iloc[0]["SETTLE_USD_T"]) if len(prompt) else float("nan")
tdl_strip = front.iloc[0]["STRIP"] if len(front) else "—"
dfn_strip = prompt.iloc[0]["STRIP"] if len(prompt) else "—"
display(Markdown(
    f"ICE session **{tdl_day}**. TDL front `{tdl_strip}` "
    f"**{tdl_px:.2f} $/t**. DFN last `{dfn_strip}` **{dfn_px:.2f} $/t**."
))

kpler_v1_auth True


ICE session **2026-10-05**. TDL front `Oct26` **238.29 $/t**. DFN last `Oct-05-26` **250.73 $/t**.

## A — Tonnage matcher: VLCCs that can reach the Mideast Gulf in 7 days

This is **reachable load-region supply**, not world fleet. Down vs 90d z-score → list is tight → near FFAs should bid if cargoes are unchanged.

In [3]:
def cached(name, fn):
    return kp._series_cache(CACHE, name, REFRESH, fn)

meg = cached(
    "kpler_tonnage_meg_vlcc_7d_11kn",
    lambda: kp.tonnage_count_series(cfg, kp.MEG_ZONE, START),
)
meg = meg.sort_values("Date") if not meg.empty else meg
if meg.empty or "vlcc_count" not in meg.columns:
    display(Markdown("_No MEG VLCC matcher rows — check zone name / entitlements._"))
else:
    meg["z90"] = kp.zscore(meg["vlcc_count"])
    display(meg.tail(3))
    tdl_d = tdl.copy()
    tdl_d["TRADE_DATE"] = pd.to_datetime(tdl_d["TRADE_DATE"])
    front_hist = (
        tdl_d.sort_values(["TRADE_DATE", "CONTRACT_MONTH"])
        .groupby("TRADE_DATE", as_index=False)
        .first()
    )
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=meg["Date"], y=meg["vlcc_count"], name="VLCC able to MEG (7d/11kn)"), secondary_y=False)
    fig.add_trace(go.Scatter(x=front_hist["TRADE_DATE"], y=front_hist["SETTLE_USD_T"], name="TDL front $/t", line=dict(dash="dot")), secondary_y=True)
    fig.update_layout(title="Reachable MEG VLCC vs TDL front", template="plotly_white", height=420, legend=dict(orientation="h"))
    fig.update_yaxes(title_text="ships", secondary_y=False)
    fig.update_yaxes(title_text="$/t", secondary_y=True)
    fig.show()
    figz = go.Figure(go.Scatter(x=meg["Date"], y=meg["z90"], name="z90"))
    figz.add_hline(y=0, line_width=1)
    figz.add_hline(y=-1, line_dash="dot")
    figz.add_hline(y=1, line_dash="dot")
    figz.update_layout(title="MEG reachable VLCC — 90d z-score (negative = tight)", template="plotly_white", height=320)
    figz.show()
    last = meg.dropna(subset=["z90"]).iloc[-1] if meg["z90"].notna().any() else meg.iloc[-1]
    ztxt = f"{last['z90']:.2f}" if pd.notna(last.get("z90")) else "n/a"
    display(Markdown(f"Latest **{pd.Timestamp(last['Date']).date()}**: {last['vlcc_count']:.0f} VLCCs, z90 **{ztxt}**."))

,Date,vlcc_count,z90
178,2026-10-04,208,1.13
179,2026-10-05,215,1.82
180,2026-10-06,222,2.45


Latest **2026-10-06**: 222 VLCCs, z90 **2.45**.

## E — Laden vs ballast (global VLCC) and China waiting

**Fleet util Ballast** = how many VLCCs are empty worldwide (~hundreds). That is the spare-supply read for tool E.

**Ballast capacity** (secondary) = Kpler port-call *capacity* in cargo tonnes for empty VLCCs — not a ship count; sparse days are normal.

China queues eat the *next* load window even if the world fleet looks large. Global ballast is a coarse proxy; MEG tightness still comes from the matcher (A).

In [4]:
util = cached("kpler_util_vlcc_state", lambda: kp.fleet_util_vlcc(cfg, START))
# Capacity (kt), not Count — see ballast_capacity_vlcc docstring / SDK docs.
bal_cap = cached(
    "kpler_ballast_capacity_vlcc",
    lambda: kp.ballast_capacity_vlcc(cfg, START),
)
cng = cached("kpler_congestion_china_vlcc", lambda: kp.congestion_china_vlcc(cfg, START))

fig = go.Figure()
for col in [c for c in util.columns if c != "Date"]:
    fig.add_trace(go.Scatter(x=util["Date"], y=util[col], name=str(col), stackgroup="one"))
fig.update_layout(title="Global VLCC by state (count)", template="plotly_white", height=400, legend=dict(orientation="h"))
fig.show()

fig2 = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
    subplot_titles=(
        "Global VLCC in ballast (util count) vs ballast capacity (t)",
        "China VLCC waiting (weekly)",
    ),
)
if not util.empty and "Ballast" in util.columns:
    fig2.add_trace(
        go.Scatter(x=util["Date"], y=util["Ballast"], name="util Ballast (ships)"),
        row=1,
        col=1,
        secondary_y=False,
    )
if not bal_cap.empty and "vlcc_ballast_capacity" in bal_cap.columns:
    fig2.add_trace(
        go.Scatter(
            x=bal_cap["Date"],
            y=bal_cap["vlcc_ballast_capacity"],
            name="ballast capacity (t)",
            line=dict(dash="dot"),
        ),
        row=1,
        col=1,
        secondary_y=True,
    )
if not cng.empty and "vlcc_waiting" in cng.columns:
    fig2.add_trace(go.Bar(x=cng["Date"], y=cng["vlcc_waiting"], name="waiting"), row=2, col=1)
fig2.update_yaxes(title_text="ships", row=1, col=1, secondary_y=False)
fig2.update_yaxes(title_text="capacity t", row=1, col=1, secondary_y=True)
fig2.update_layout(template="plotly_white", height=560, legend=dict(orientation="h"))
fig2.show()

## B — Fixtures MEG → China VLCC (counts only)

Kpler `size` caps the pull (default 1000). This is **reported** fixtures, not Baltic. Median `$` is Kpler’s print — colour only.

In [5]:
fx = cached(
    "kpler_fix_meg_china_vlcc_counts",
    lambda: kp.fixtures_vlcc_summary(cfg, START),
)
display(fx.tail(10))
fig = make_subplots(specs=[[{"secondary_y": True}]])
if not fx.empty:
    fig.add_trace(go.Bar(x=fx["Date"], y=fx["n_fixtures"], name="n fixtures"), secondary_y=False)
    fig.add_trace(go.Scatter(x=fx["Date"], y=fx["median_rate"], name="median Kpler $"), secondary_y=True)
fig.update_layout(title="MEG→China VLCC fixtures (Kpler)", template="plotly_white", height=380, legend=dict(orientation="h"))
fig.update_yaxes(title_text="count", secondary_y=False)
fig.update_yaxes(title_text="Kpler $", secondary_y=True)
fig.show()
n_last = int(fx.iloc[-1]["n_fixtures"]) if len(fx) else 0
display(Markdown(f"{len(fx)} days with ≥1 fixture in the pull. Last day n={n_last}."))

,Date,n_fixtures,median_rate
13,2026-07-08,3,270.00
14,2026-07-15,1,NaN
15,2026-08-05,1,465.00
16,2026-08-10,1,NaN
17,2026-08-18,1,NaN
18,2026-08-19,1,570.00
19,2026-09-08,1,650.00
20,2026-09-25,3,NaN
21,2026-09-28,1,NaN
22,2026-10-05,1,NaN


23 days with ≥1 fixture in the pull. Last day n=1.

## F — VLCC ton-miles (monthly)

Regime, not a one-week trade. Up with flat volumes = longer-haul (Atlantic to Asia).

Pulled as `FreightMetricsSeries` split by **vesselTypeOil** → VLCC column (`Total` + VLCC filter returns empty on this account).

In [6]:
tm_start = date.today().replace(year=date.today().year - 3)
# New cache key after VesselTypeOil fix (old parquet had empty Total).
tm = cached("kpler_tonmiles_vlcc_m_vto", lambda: kp.tonmiles_vlcc(cfg, tm_start))
fig = go.Figure()
if not tm.empty and tm["ton_miles"].notna().any():
    fig.add_trace(go.Bar(x=tm["Date"], y=tm["ton_miles"], name="kt·nmi"))
    display(Markdown(f"Last non-null: **{tm.dropna(subset=['ton_miles']).iloc[-1]['Date'].date()}** "
                     f"= {tm.dropna(subset=['ton_miles']).iloc[-1]['ton_miles']:,.0f} kt·nmi."))
else:
    display(Markdown("_No VLCC ton-miles — check FreightMetrics entitlement / split._"))
fig.update_layout(title="VLCC ton-miles (monthly, freight view on)", template="plotly_white", height=360)
fig.show()

Last non-null: **2026-10-01** = 93,687,345,536 kt·nmi.

## F2 — VLCC ton-miles → China (seasonality + origins)

Same chart grammar as clean Latam desks use, applied to the TD3C discharge basin:

1. **Seasonal:** monthly VLCC ton-miles into China vs prior 5-year range/mean (years overlaid).
2. **By origin:** MEG / West Africa / Brazil / US / Russia / Other — who is supplying China haulage.

Units shown as **Bn** (`value / 1e9`). Global VLCC ton-miles (section F) stays the blunt world tape; this is the China corridor. Origin pulls use `from_locations` + vesselTypeOil (OriginCountry×VLCC is empty on this key).

In [7]:
tm_china_start = date(2020, 1, 1)
tm_china = cached(
    "kpler_tonmiles_vlcc_china_m",
    lambda: kp.tonmiles_vlcc_to(cfg, "China", tm_china_start),
)
tm_china_bn = tm_china.assign(bn=tm_china["ton_miles"] / kp.TONMILES_BN) if not tm_china.empty else tm_china

seas = kp.seasonal_tonmiles(tm_china_bn, value_col="bn", history_years=5)
band, years = seas["band"], seas["years"]
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig_s = go.Figure()
if not band.empty:
    fig_s.add_trace(go.Scatter(
        x=band["month"], y=band["y5_hi"], name="5y high",
        line=dict(width=0), showlegend=False,
    ))
    fig_s.add_trace(go.Scatter(
        x=band["month"], y=band["y5_lo"], name="5y range",
        fill="tonexty", fillcolor="rgba(180,180,180,0.35)", line=dict(width=0),
    ))
    fig_s.add_trace(go.Scatter(
        x=band["month"], y=band["y5_mean"], name="5y mean",
        line=dict(color="crimson", dash="dot"),
    ))
if not years.empty:
    for col in [c for c in years.columns if c != "month"]:
        fig_s.add_trace(go.Scatter(x=years["month"], y=years[col], name=str(col)))
fig_s.update_layout(
    title="VLCC ton-miles to China — seasonal (Bn)",
    template="plotly_white", height=420, legend=dict(orientation="h"),
    xaxis=dict(tickmode="array", tickvals=list(range(1, 13)), ticktext=month_labels),
    yaxis_title="Bn",
)
fig_s.show()

if not tm_china_bn.empty and tm_china_bn["bn"].notna().any():
    last = tm_china_bn.dropna(subset=["bn"]).iloc[-1]
    display(Markdown(
        f"China VLCC ton-miles last month **{pd.Timestamp(last['Date']).strftime('%Y-%m')}**: "
        f"**{last['bn']:.1f} Bn**. Compare to 5y mean for that calendar month."
    ))

China VLCC ton-miles last month **2026-10**: **33.9 Bn**. Compare to 5y mean for that calendar month.

In [8]:
origin_start = date(2022, 1, 1)
tm_orig = cached(
    "kpler_tonmiles_vlcc_china_by_origin_m",
    lambda: kp.tonmiles_vlcc_to_by_origin(cfg, "China", origin_start),
)

fig_o = go.Figure()
if not tm_orig.empty:
    for col in [c for c in tm_orig.columns if c != "Date"]:
        fig_o.add_trace(go.Scatter(
            x=tm_orig["Date"], y=tm_orig[col] / kp.TONMILES_BN, name=col,
        ))
fig_o.update_layout(
    title="VLCC ton-miles to China by origin (Bn)",
    template="plotly_white", height=420, legend=dict(orientation="h"),
    yaxis_title="Bn",
)
fig_o.show()

if not tm_orig.empty:
    last = tm_orig.iloc[-1]
    named = [c for c in tm_orig.columns if c not in ("Date", "Other")]
    total = sum(float(last[c]) for c in named + ["Other"] if pd.notna(last.get(c)))
    if total > 0:
        meg_share = 100.0 * float(last["Mideast Gulf"]) / total if pd.notna(last.get("Mideast Gulf")) else float("nan")
        display(Markdown(
            f"Latest month MEG share of named+Other China VLCC ton-miles: **{meg_share:.0f}%**. "
            f"Rising Atlantic (WAF/Brazil/US) with flat MEG = ships pulled west of Suez — softer TD3C relative to Atlantic routes."
        ))

Latest month MEG share of named+Other China VLCC ton-miles: **35%**. Rising Atlantic (WAF/Brazil/US) with flat MEG = ships pulled west of Suez — softer TD3C relative to Atlantic routes.

## Fleet development (v2 key) — year story, not the nowcast

Monthly **all-fleet** `available` count from v2 series. Summary KPI on this key returns stale 2019–20 periods and cannot filter VLCC — do **not** use it for front TDL. This chart is structural context only.

In [9]:
try:
    fd = cached(
        "kpler_fleet_dev_available_m",
        lambda: kp.fleet_development_available_series_v2(ROOT),
    )
    # Series includes forward projections; history for context, as-of today for the tile.
    today = pd.Timestamp(date.today())
    fd_hist = fd.loc[(fd["Date"] >= pd.Timestamp("2022-01-01")) & (fd["Date"] <= today)] if not fd.empty else fd
    fd_fwd = fd.loc[fd["Date"] > today] if not fd.empty else fd
    fig = go.Figure()
    if not fd_hist.empty:
        fig.add_trace(go.Scatter(x=fd_hist["Date"], y=fd_hist["available_count"], name="available (history)"))
        if not fd_fwd.empty:
            fig.add_trace(go.Scatter(
                x=fd_fwd["Date"], y=fd_fwd["available_count"],
                name="available (projection)", line=dict(dash="dot"),
            ))
        last = fd_hist.dropna(subset=["available_count"]).iloc[-1]
        display(Markdown(
            f"As-of **{last['Date'].date()}**: available count **{last['available_count']:,.0f}** "
            f"(all fleet types — not VLCC). Forward points are orderbook projection."
        ))
    else:
        display(Markdown("_No v2 fleet-development series._"))
    fig.update_layout(
        title="Fleet development — available count (v2, all fleet)",
        template="plotly_white",
        height=320,
    )
    fig.show()
except Exception as exc:
    display(Markdown(f"v2 fleet-development skipped: `{type(exc).__name__}: {exc}`"))

As-of **2026-10-01**: available count **32,853** (all fleet types — not VLCC). Forward points are orderbook projection.

## C — Clarksons-style tape (Kpler proxies)

Inspired by the 06 Oct 2026 Clarksons VLCC pack (availability + China analysis). **Not a reprint** — methodology check only; do not treat as Clarksons data.

| Their slide | Our proxy |
|---|---|
| MEG availability 15/20/25/30d vs TD3C | TonnageSupply ladder `eta` 7–30 @ 11 kn vs TDL $/t |
| Non-restricted laden/ballast | **All-VLCC** util Loaded/(Loaded+Ballast) — no sanction/dark filter |
| Atlantic / MEG fixture volumes | Monthly fixture **counts** (MEG spot all dest; Atlantic→China) |

Still missing vs that pack: Baltic WS/TCE, FOC-west, East/West location %, China arrived/sailed 5d, floating storage, MEG liftings mb, restricted fleet.

In [10]:
ladder_start = date.today() - timedelta(days=365)
ladder = cached(
    "kpler_tonnage_meg_ladder_7_30",
    lambda: kp.tonnage_ladder(cfg, kp.MEG_ZONE, ladder_start, etas=kp.MEG_AVAIL_ETAS),
)

tdl_d = tdl.copy()
tdl_d["TRADE_DATE"] = pd.to_datetime(tdl_d["TRADE_DATE"])
front_hist = (
    tdl_d.sort_values(["TRADE_DATE", "CONTRACT_MONTH"])
    .groupby("TRADE_DATE", as_index=False)
    .first()
)

fig_l = make_subplots(specs=[[{"secondary_y": True}]])
eta_cols = [c for c in ladder.columns if c.startswith("eta_")]
for col in eta_cols:
    fig_l.add_trace(
        go.Scatter(x=ladder["Date"], y=ladder[col], name=col.replace("_", " ")),
        secondary_y=False,
    )
fig_l.add_trace(
    go.Scatter(
        x=front_hist["TRADE_DATE"], y=front_hist["SETTLE_USD_T"],
        name="TDL front $/t", line=dict(dash="dot", color="black"),
    ),
    secondary_y=True,
)
fig_l.update_layout(
    title="MEG VLCC availability ladder (Kpler matcher) vs TDL",
    template="plotly_white", height=440, legend=dict(orientation="h"),
)
fig_l.update_yaxes(title_text="ships", secondary_y=False)
fig_l.update_yaxes(title_text="$/t", secondary_y=True)
fig_l.show()

if not ladder.empty and any(ladder[c].notna().any() for c in eta_cols):
    last = ladder.dropna(subset=eta_cols, how="all").iloc[-1]
    bits = ", ".join(
        f"{c.replace('eta_', '')}d={int(last[c])}" for c in eta_cols if pd.notna(last[c])
    )
    display(Markdown(
        f"Latest **{pd.Timestamp(last['Date']).date()}**: {bits}. "
        f"Wider horizons ≫ 7d = distant ballast not yet in the prompt list."
    ))
elif not ladder.empty:
    display(Markdown("_Ladder rows present but all eta columns empty._"))
else:
    display(Markdown("_No MEG ladder data._"))

Latest **2026-10-06**: 7d=221, 15d=481, 20d=543, 25d=645, 30d=753. Wider horizons ≫ 7d = distant ballast not yet in the prompt list.

In [11]:
# Reuse util cache from section E when present; else fetch.
if "util" not in dir() or util is None or getattr(util, "empty", True):
    util = cached("kpler_util_vlcc_state", lambda: kp.fleet_util_vlcc(cfg, START))
lbr = kp.laden_ballast_ratio(util)

fig_r = make_subplots(specs=[[{"secondary_y": True}]])
if not lbr.empty:
    fig_r.add_trace(
        go.Scatter(x=lbr["Date"], y=lbr["Loaded"], name="Loaded", stackgroup="lb"),
        secondary_y=False,
    )
    fig_r.add_trace(
        go.Scatter(x=lbr["Date"], y=lbr["Ballast"], name="Ballast", stackgroup="lb"),
        secondary_y=False,
    )
    fig_r.add_trace(
        go.Scatter(
            x=lbr["Date"], y=lbr["laden_ballast_ratio"],
            name="laden/(laden+ballast)", line=dict(color="black", dash="dot"),
        ),
        secondary_y=True,
    )
fig_r.update_layout(
    title="VLCC laden vs ballast (all Liquids VLCC — not Clarksons non-restricted)",
    template="plotly_white", height=400, legend=dict(orientation="h"),
)
fig_r.update_yaxes(title_text="ships", secondary_y=False)
fig_r.update_yaxes(title_text="ratio", range=[0, 1], secondary_y=True)
fig_r.show()

if not lbr.empty and lbr["laden_ballast_ratio"].notna().any():
    last = lbr.dropna(subset=["laden_ballast_ratio"]).iloc[-1]
    display(Markdown(
        f"Latest **{pd.Timestamp(last['Date']).date()}**: ratio **{last['laden_ballast_ratio']:.2f}** "
        f"(Loaded {last['Loaded']:.0f} / Ballast {last['Ballast']:.0f}). "
        f"Higher ratio = fleet more committed."
    ))

Latest **2026-10-06**: ratio **0.42** (Loaded 383 / Ballast 520). Higher ratio = fleet more committed.

In [12]:
fx_m_start = date.today().replace(year=date.today().year - 2)
fx_m = cached(
    "kpler_fix_corridor_monthly",
    lambda: kp.fixtures_corridor_monthly(cfg, fx_m_start),
)

fig_f = go.Figure()
if not fx_m.empty:
    if "meg_spot" in fx_m.columns:
        fig_f.add_trace(go.Bar(x=fx_m["Date"], y=fx_m["meg_spot"], name="MEG spot (all dest)"))
    if "atlantic_to_china" in fx_m.columns:
        fig_f.add_trace(go.Bar(x=fx_m["Date"], y=fx_m["atlantic_to_china"], name="Atlantic→China"))
fig_f.update_layout(
    title="VLCC fixture counts — MEG spot vs Atlantic→China (monthly)",
    template="plotly_white", height=400, barmode="group", legend=dict(orientation="h"),
    yaxis_title="n fixtures",
)
fig_f.show()
display(fx_m.tail(6) if not fx_m.empty else Markdown("_No corridor fixture months._"))
display(Markdown(
    "Atlantic = sum of West Africa / Brazil / United States → China. "
    "`size=1000` can truncate busy months — treat as activity colour, not a census."
))

,Date,meg_spot,atlantic_to_china,West Africa,Brazil,United States
19,2026-05-01,21.00,47.00,19,18,10.00
20,2026-06-01,72.00,53.00,24,23,6.00
21,2026-07-01,120.00,48.00,15,23,10.00
22,2026-08-01,70.00,57.00,14,22,21.00
23,2026-09-01,107.00,78.00,35,23,20.00
24,2026-10-01,12.00,5.00,1,4,NaN


Atlantic = sum of West Africa / Brazil / United States → China. `size=1000` can truncate busy months — treat as activity colour, not a census.

### Still cannot replicate from this stack

- **TD3C/TD20 WS & TCE $/day** — need Baltic (or locked Platts≡Baltic) + bunker/voyage TCE model; ICE is $/t.
- **Ballast/FOC committed west** — next-destination 403; FOC is Clarksons.
- **East/West ballast % and location breakdown** — need vessel positions by basin.
- **China arrived/sailed last 5d / laden &gt;14d** — PortCalls + their China methodology (congestion ≠ arrivals).
- **Non-restricted / dark fleet** — sanction tags (possible later via `vessel_crossing`, not Liquids util).
- **MEG liftings mb, floating storage, removals scenarios** — Trades aggregation / storage definition / broker judgment.

## How to read this vs paper

| If you see | Read |
|---|---|
| MEG reachable z-score ≪ 0 and fixtures firm | Physical tight — TDL/DFN rich unless paper already caught up |
| Ballast rising, matcher rising | List rebuilding — fade a spike |
| China waiting up, matcher not rising | Ships stuck at discharge — MEG can stay tight |
| Ton-miles up, MEG list long | Atlantic/long-haul, not necessarily TD3C |
| China VLCC ton-miles ≪ 5y mean | Soft discharge demand / shorter hauls — less ship-week absorption |
| WAF/Brazil/US share up vs MEG into China | Atlantic pulls VLCCs; TD3C can lag Atlantic routes |
| Ladder 7d tight but 30d fat | Prompt list short; ballast still inbound — spike may fade |
| Laden/ballast ratio rising | Fleet more committed — supportive for prompt if MEG list also tight |

Next destination / Routes / Kpler arb / voyage-legs are **not entitled** on this v2 key. For backtests use matcher series history and `TradesSnapshot` on v1 — not live fixture dumps.